In [ ]:
# =========================
# Imports & setup
# =========================
import numpy as np
import matplotlib.pyplot as plt
# (Optional) seaborn is fine in notebooks; comment out if running headless
import seaborn as sns
from tqdm import tqdm
from scipy.optimize import curve_fit
from scipy.interpolate import RegularGridInterpolator
import pandas as pd

# Optional styling (safe to remove outside notebooks)
sns.set_theme(style="whitegrid")
# %matplotlib inline  # keep only if you're in a Jupyter notebook

# =========================
# Your geometry helpers (unchanged)
# =========================
def create_circular_electrode_mask(cx, cy, radius, X, Y):
    dist_from_center = np.sqrt((X - cx)**2 + (Y - cy)**2)
    mask = dist_from_center <= radius
    return mask

def create_rectangular_electrode_mask(cx, cy, width, height, X, Y):
    x1 = cx - width / 2
    x2 = cx + width / 2
    y1 = cy - height / 2
    y2 = cy + height / 2
    mask = (X >= x1) & (X <= x2) & (Y >= y1) & (Y <= y2)
    return mask

def create_rotated_hyperbolic_electrode_mask(x0, y0, a, b, theta_deg, X, Y, branch='positive'):
    theta = np.radians(theta_deg)
    x_shift = X - x0
    y_shift = Y - y0
    u = x_shift * np.cos(theta) + y_shift * np.sin(theta)
    v = -x_shift * np.sin(theta) + y_shift * np.cos(theta)
    lhs = (u**2) / a**2 - (v**2) / b**2
    if branch == 'positive':
        mask = lhs >= 1
    elif branch == 'negative':
        mask = lhs <= -1
    else:
        raise ValueError("branch must be 'positive' or 'negative'")
    return mask

def solve_potential(nx, ny, electrodes, fixed, V, max_iter=10000, tol=1e-3):
    for it in tqdm(range(max_iter)):
        V_old = V.copy()
        for y in range(1, ny - 1):
            for x in range(1, nx - 1):
                if not fixed[y, x]:
                    V[y, x] = 0.25 * (V[y + 1, x] + V[y - 1, x] + V[y, x + 1] + V[y, x - 1])
        # Reapply fixed voltages after update
        for electrode in electrodes:
            mask = electrode['mask']
            V[mask] = electrode['voltage']
        diff = np.max(np.abs(V - V_old))
        if diff < tol:
            print(f'Converged after {it} iterations.')
            break
    else:
        print(f'Max iterations reached, diff={diff}')
    return V

def compute_electric_field(V, dx=1.0, dy=1.0):
    Ey, Ex = np.gradient(V, dy, dx)
    return -Ex, -Ey

def plot_mask_outline(ax, mask, X, Y, linewidth=1.5, label=None):
    cs = ax.contour(X, Y, mask.astype(float), levels=[0.5], colors='red', linewidths=linewidth)
    if label is not None:
        ys, xs = np.where(mask)
        if len(xs) > 0:
            cx = X[0, xs].mean()
            cy = Y[ys, 0].mean()
            ax.text(cx, cy, label, color='red', ha='center', va='center', fontsize=8, fontweight='bold')

def plot_potential_and_field_side_by_side(V, Ex, Ey, X, Y, electrodes, step=4):
    fig, axes = plt.subplots(1, 2, figsize=(20, 6), sharex=True, sharey=True)
    ax1 = axes[0]
    im1 = ax1.imshow(V, extent=[X.min(), X.max(), Y.min(), Y.max()], origin='lower', cmap='viridis')
    fig.colorbar(im1, ax=ax1, label='Potential (V)')
    ax1.set_title("Electric Potential")
    ax1.set_xlabel("x (μm)")
    ax1.set_ylabel("y (μm)")

    ax2 = axes[1]
    im2 = ax2.imshow(V, extent=[X.min(), X.max(), Y.min(), Y.max()], origin='lower', cmap='viridis')
    fig.colorbar(im2, ax=ax2, label='Potential (V)')
    ax2.quiver(X[::step, ::step], Y[::step, ::step], Ex[::step, ::step], Ey[::step, ::step], color='white', scale=100)
    ax2.set_title("Electric Field Vectors")
    ax2.set_xlabel("x (μm)")

    for ax in axes:
        for e in electrodes:
            label = f"{e['voltage']} V" if 'voltage' in e else None
            plot_mask_outline(ax, e['mask'], X, Y, label=label)
        ax.set_aspect('equal')

    plt.tight_layout()
    plt.show()

def quadratic_fit(s, a, c):
    return a * s**2 + c

def get_cut_along_angle(V, X, Y, angle_deg, length):
    angle_rad = np.deg2rad(angle_deg)
    s = np.linspace(-length, length, 2 * length + 1)  # 1 μm steps
    x_cut = s * np.cos(angle_rad)
    y_cut = s * np.sin(angle_rad)
    interp_func = RegularGridInterpolator((Y[:,0], X[0,:]), V)
    points = np.vstack((y_cut, x_cut)).T  # order (y, x)
    V_cut = interp_func(points)
    return s, V_cut, x_cut, y_cut

def multipole_func(coords, ion_elc_dis, pn_rs, pn_is):
    if len(pn_rs) != len(pn_is):
        return None
    X, Y = coords
    n_order = len(pn_rs)
    potential = 0
    for i in range(n_order):
        multipole_expansion = (X + 1j * Y)**i
        pn_real_func = np.real(multipole_expansion)
        pn_imag_func = np.imag(multipole_expansion)
        potential += 1/ion_elc_dis**i * (pn_rs[i] * pn_real_func + pn_is[i] * pn_imag_func)
    return potential

# =========================
# NEW: Multipole fitting utilities
# =========================
def _harmonic_basis_xy(x, y, n):
    z = x + 1j*y
    zn = z**n
    return np.real(zn), np.imag(zn)

def fit_multipoles_2d(V, X, Y, *, r_fit_um=100.0, ion_elc_dis_um=50.0, n_max=8, exclude_mask=None):
    """
    Fit V(x,y) inside radius r_fit_um to:
        V ≈ sum_{n=0..n_max} (1/R^n) [ a_n Re((x+iy)^n) + b_n Im((x+iy)^n) ],
    where R = ion_elc_dis_um.
    Returns dict with a[n], b[n], p[n] = sqrt(a_n^2 + b_n^2).
    """
    x = X.ravel(); y = Y.ravel(); v = V.ravel()
    r = np.hypot(x, y)
    region = (r <= r_fit_um)
    if exclude_mask is not None:
        region &= ~exclude_mask.ravel()
    x_sel, y_sel, v_sel = x[region], y[region], v[region]

    cols = []
    for n in range(n_max + 1):
        Re, Im = _harmonic_basis_xy(x_sel, y_sel, n)
        scale = ion_elc_dis_um**(-n)
        cols.append(scale * Re)
        cols.append(scale * Im)
    A = np.vstack(cols).T  # (N_pts, 2*(n_max+1))

    coeffs, *_ = np.linalg.lstsq(A, v_sel, rcond=None)
    a, b, p = {}, {}, {}
    for n in range(n_max + 1):
        a[n] = float(coeffs[2*n])
        b[n] = float(coeffs[2*n + 1])
        p[n] = float(np.hypot(a[n], b[n]))
    return {"a": a, "b": b, "p": p}

def multipole_ratio_row(p_dict, base_n=2, max_n=8):
    """
    Build a dict of ratios p_n/p_base for n=3..max_n.
    """
    p2 = p_dict.get(base_n, np.nan)
    row = {}
    for n in range(3, max_n + 1):
        row[f"p{n}/p{base_n}"] = (p_dict.get(n, np.nan) / p2) if (p2 and not np.isnan(p2)) else np.nan
    return row

# =========================
# Problem setup (your parameters)
# =========================
nx, ny = 150, 150
dx = 4.0  # μm
x = (np.arange(nx) - nx//2) * dx
y = (np.arange(ny) - ny//2) * dx
X, Y = np.meshgrid(x, y)

x_sep = 0
y_sep = 0
a = 280.0
b = 280.0
electrodes = [
    {'center': (-x_sep,  y_sep), 'dims': (a, b), 'theta': -45, 'branch': 'positive', 'voltage':  1.0},
    {'center': ( x_sep,  y_sep), 'dims': (a, b), 'theta':  45, 'branch': 'positive', 'voltage': -1.0},
    {'center': (-x_sep, -y_sep), 'dims': (a, b), 'theta': -45, 'branch': 'negative', 'voltage': -1.0},
    {'center': ( x_sep, -y_sep), 'dims': (a, b), 'theta':  45, 'branch': 'negative', 'voltage':  1.0},
]

V = np.zeros((ny, nx))
fixed = np.zeros_like(V, dtype=bool)

for e in electrodes:
    cx, cy = e['center']
    aa, bb = e['dims']
    th = e['theta']
    branch = e['branch']
    mask = create_rotated_hyperbolic_electrode_mask(cx, cy, aa, bb, th, X, Y, branch=branch)
    e['mask'] = mask
    V[mask] = e['voltage']
    fixed[mask] = True

# =========================
# Solve, fields, and your original plots
# =========================
V = solve_potential(nx, ny, electrodes, fixed, V, max_iter=10000, tol=1e-5)
Ex, Ey = compute_electric_field(V, dx=dx, dy=dx)
plot_potential_and_field_side_by_side(V, Ex, Ey, X, Y, electrodes)

# =========================
# Your diagonal cuts (unchanged)
# =========================
length = 100  # μm half-length of cut
angles = [-45, 45]

plt.figure(figsize=(5, 10))
extent = [X.min(), X.max(), Y.min(), Y.max()]
plt.subplot(3,1,1)
im = plt.imshow(V, origin='lower', extent=extent, cmap='viridis')
plt.colorbar(im, label='Potential (V)')
plt.xlabel('x (μm)')
plt.ylabel('y (μm)')
plt.title('Potential with Diagonal Cuts')

for angle in angles:
    angle_rad = np.deg2rad(angle)
    L = max(X.max() - X.min(), Y.max() - Y.min())
    s_line = np.linspace(-L/2, L/2, 500)
    x_line = s_line * np.cos(angle_rad)
    y_line = s_line * np.sin(angle_rad)
    plt.plot(x_line, y_line, 'w--', lw=2, label=f'{angle}° cut')
plt.legend()

def quadratic_fit(s, a, c):
    return a * s**2 + c

def get_cut_along_angle(V, X, Y, angle_deg, length):
    angle_rad = np.deg2rad(angle_deg)
    s = np.linspace(-length, length, 2 * length + 1)  # 1 μm steps
    x_cut = s * np.cos(angle_rad)
    y_cut = s * np.sin(angle_rad)
    interp_func = RegularGridInterpolator((Y[:,0], X[0,:]), V)
    points = np.vstack((y_cut, x_cut)).T  # order (y, x)
    V_cut = interp_func(points)
    return s, V_cut, x_cut, y_cut

for i, angle in enumerate(angles):
    s, V_cut, x_cut, y_cut = get_cut_along_angle(V, X, Y, angle, length)
    popt, pcov = curve_fit(quadratic_fit, s, V_cut)
    a_fit, c_fit = popt
    electro_dis = 50.0  # μm, your choice used in geom factor readout
    print(f'Geometric factor for {angle}°:', np.round(np.abs(a_fit * electro_dis**2), 4))
    plt.subplot(3,1,i+2)
    plt.plot(s, V_cut, 'b.', label='Data')
    plt.plot(s, quadratic_fit(s, *popt), 'r-', label=f'Fit: a={a_fit:.3e} V/μm²')
    plt.title(f'Cut at {angle}°')
    plt.xlabel('Distance s (μm)')
    plt.ylabel('Potential V (V)')
    plt.grid(True)
    plt.legend()
    print(f"Angle {angle}° fit: a = {a_fit:.3e} V/μm², offset = {c_fit:.3e} V")

plt.tight_layout()
plt.show()

# =========================
# NEW: Build multipole ratio table for YOUR geometry
# =========================
# Exclude electrode metal when fitting
exclude_mask = np.zeros_like(V, dtype=bool)
for e in electrodes:
    exclude_mask |= e['mask']

# Fit multipoles around the center
fit = fit_multipoles_2d(
    V, X, Y,
    r_fit_um=100.0,       # fit radius (μm) around origin
    ion_elc_dis_um=50.0,  # R used to normalize p_n (same scale you used)
    n_max=8,
    exclude_mask=exclude_mask
)

your_row = multipole_ratio_row(fit["p"], base_n=2, max_n=8)
your_df = pd.DataFrame([your_row], index=["Your hyperbolic geometry"])

# =========================
# NEW: (Optional) Append the literature rows you pasted for comparison
# =========================
# NOTE: These are taken exactly from your pasted table; adjust names if desired.
literature_rows = {
    "2-layer":                 {"p3/p2":0.0015,"p4/p2":0.0284,"p5/p2":0.0005,"p6/p2":0.1171,"p7/p2":0.0036,"p8/p2":0.0039},
    "balanced 2-layer":        {"p3/p2":0.0011,"p4/p2":0.0158,"p5/p2":0.0009,"p6/p2":0.1156,"p7/p2":0.0005,"p8/p2":0.0242},
    "3-layer":                 {"p3/p2":0.0006,"p4/p2":0.1904,"p5/p2":0.0006,"p6/p2":0.0554,"p7/p2":0.0005,"p8/p2":0.0203},
    "2-layer AlGaAs":          {"p3/p2":0.0026,"p4/p2":0.6005,"p5/p2":0.0054,"p6/p2":0.4340,"p7/p2":0.0143,"p8/p2":0.2642},
    "in-plane 4-wire":         {"p3/p2":0.0008,"p4/p2":0.3443,"p5/p2":0.0009,"p6/p2":0.1191,"p7/p2":0.0010,"p8/p2":0.0330},
    "4-wire surface":          {"p3/p2":0.9731,"p4/p2":0.6931,"p5/p2":0.4255,"p6/p2":0.2442,"p7/p2":0.1481,"p8/p2":0.1074},
    "5-wire symm. surface":    {"p3/p2":0.9963,"p4/p2":0.6202,"p5/p2":0.2640,"p6/p2":0.0408,"p7/p2":0.1504,"p8/p2":0.1426},
    "5-wire asymm. surface":   {"p3/p2":1.0100,"p4/p2":0.6401,"p5/p2":0.2737,"p6/p2":0.0569,"p7/p2":0.1195,"p8/p2":0.1455},
}
lit_df = pd.DataFrame.from_dict(literature_rows, orient="index")

# Align columns and combine
all_cols = [f"p{n}/p2" for n in range(3,9)]
your_df = your_df.reindex(columns=all_cols)
lit_df = lit_df.reindex(columns=all_cols)
result_df = pd.concat([your_df, lit_df], axis=0)

# Show + save
print("\nMultipole ratios (p_n / p_2):")
print(result_df.to_string(float_format=lambda v: f"{v:.6f}"))

out_csv = "multipole_ratios_comparison.csv"
result_df.to_csv(out_csv)
print(f"\nSaved table to: {out_csv}")
